In [2]:
# Import necessary libraries
import pandas as pd
from pathlib import Path

In [3]:
# Pipe constants
pipe_length = 50.0
pipe_diameter = 0.2

# Sensor positions
NODE_A_X = 5.0
NODE_B_X = 25.0
NODE_C_X = 45.0
tolerance = 0.5

feature_columns = [
    "pressure",
    "velocity-magnitude",
    "turb-kinetic-energy",
    "turb-diss-rate",
    "wall-shear",
    "tailings-vof"
]

raw_data_path = Path.cwd().parent.parent / "data" / "raw"
synthetic_path = Path.cwd().parent.parent / "data" / "synthetic"
output_path = Path.cwd().parent.parent / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)

print("CONFIGURATION")
print(f"Node A: {NODE_A_X}, Node B: {NODE_B_X}, Node C: {NODE_C_X}")
print(f"Output path: {output_path}")

CONFIGURATION
Node A: 5.0, Node B: 25.0, Node C: 45.0
Output path: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed


In [4]:
# Effect factor category
def get_effect_factor(name):
    name = name.lower()

    if "25" in name:
        return 0.25
    elif "50" in name:
        return 0.55
    elif "75" in name:
        return 0.90
    else:
        return 0.0

In [5]:
# Loading the three data files.
scenario_data = []

def load_csv_with_label(path, scenario_id, label):
    df = pd.read_csv(path)
    df["effect_factor"] = get_effect_factor(str(scenario_id))
    df["scenario"] = scenario_id
    df["label"] = label
    return df

# Normal
normal_file = raw_data_path / "normal_dataset.csv"
scenario_data.append((load_csv_with_label(normal_file, "normal_baseline", 0), "normal_baseline", 0))
# Normal variations
for file in (synthetic_path / "normal_variations").glob("*.csv"):
    if "combined" not in file.stem:
        scenario_data.append((load_csv_with_label(file, file.stem, 0), file.stem, 0))

# Leak
for file in (synthetic_path / "leakage_variations").glob("*.csv"):
    if "combined" not in file.stem:
        scenario_data.append((load_csv_with_label(file, file.stem, 1), file.stem, 1))

# Blockage
for file in (synthetic_path / "blockage_variations").glob("*.csv"):
    if "combined" not in file.stem:
        scenario_data.append((load_csv_with_label(file, file.stem, 2), file.stem, 2))
# Summary
print("SCENARIO SUMMARY")
print("Total scenarios:", len(scenario_data))
print("Normal:", sum(1 for _,_,l in scenario_data if l==0))
print("Leak:", sum(1 for _,_,l in scenario_data if l==1))
print("Blockage:", sum(1 for _,_,l in scenario_data if l==2))

SCENARIO SUMMARY
Total scenarios: 45
Normal: 15
Leak: 15
Blockage: 15


In [6]:
# Fault type label
def get_fault_type(label):
    mapping = {
        0: "Normal",
        1: "Leak",
        2: "Blockage"
    }
    return mapping.get(label, "Unknown")

In [7]:
# Extracting sensor data points from global dataset
# Finds the closest point to our sensor nodes A,B, C
# Takes measurement from there and saves
def closest_node(df_t, x_target):
    idx = (df_t["x-coordinate"] - x_target).abs().idxmin()
    return df_t.loc[idx]

def extract_sensor_readings(df_scenario, scenario_id, label):
    rows = []

    for t, df_t in df_scenario.groupby("timestep"):

        a = closest_node(df_t, NODE_A_X)
        b = closest_node(df_t, NODE_B_X)
        c = closest_node(df_t, NODE_C_X)

        effect_factor = df_t["effect_factor"].iloc[0]
        fault_type = get_fault_type(label)

        rows.append({
            "scenario_id": scenario_id,
            "timestep": t,

            "node_a_pressure": a["pressure"],
            "velocity_a": a["velocity-magnitude"],
            "tke_a": a["turb-kinetic-energy"],
            "tdr_a": a["turb-diss-rate"],
            "wall_shear_a": a["wall-shear"],
            "tailings_vof_a": a["tailings-vof"],

            "node_b_pressure": b["pressure"],
            "velocity_b": b["velocity-magnitude"],
            "tke_b": b["turb-kinetic-energy"],
            "tdr_b": b["turb-diss-rate"],
            "wall_shear_b": b["wall-shear"],
            "tailings_vof_b": b["tailings-vof"],

            "node_c_pressure": c["pressure"],
            "velocity_c": c["velocity-magnitude"],
            "tke_c": c["turb-kinetic-energy"],
            "tdr_c": c["turb-diss-rate"],
            "wall_shear_c": c["wall-shear"],
            "tailings_vof_c": c["tailings-vof"],

            "label": label,
            "effect_factor": effect_factor,
            "fault_type": fault_type
        })
    df_out = pd.DataFrame(rows)

    print(f"[OK] {scenario_id} → shape {df_out.shape}")

    return df_out

In [8]:
def calculate_derived_features(df_wide: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates derived features from wide-format sensor data.

    Derived features:
        - Pressure drops between nodes (A→B, B→C, A→C)
        - Rate of pressure change over time (dp/dt) per node, per scenario
        - Pressure gradient ratio (asymmetry indicator)
        - Velocity differential between nodes
        - TKE differential between nodes (turbulence anomaly indicator)
        - Flow velocity proxy (node B bulk velocity)

    Args:
        df_wide: Wide-format DataFrame with one row per (scenario_id, timestep)

    Returns:
        DataFrame with original columns + derived features, sorted by scenario and time
    """

    df = df_wide.copy()

    df = df.sort_values(["scenario_id", "timestep"]).reset_index(drop=True)

    print("Calculating derived features...")

    df["pressure_drop_ab"] = df["node_a_pressure"] - df["node_b_pressure"]
    df["pressure_drop_bc"] = df["node_b_pressure"] - df["node_c_pressure"]
    df["pressure_drop_ac"] = df["node_a_pressure"] - df["node_c_pressure"]

    # Ratio: imbalance between upstream and downstream pressure drop
    # Deviations from ~1.0 can signal a mid-pipe anomaly (leak or blockage)
    df["pressure_gradient_ratio"] = df["pressure_drop_ab"] / (
        df["pressure_drop_bc"].replace(0, float("nan"))
    )
    df["pressure_gradient_ratio"] = df["pressure_gradient_ratio"].fillna(0.0).clip(-10, 10)

    TIMESTEP_INTERVAL = 0.5  # seconds between timesteps

    pressure_cols = {
        "a": "node_a_pressure",
        "b": "node_b_pressure",
        "c": "node_c_pressure",
    }

    for node, pressure_col in pressure_cols.items():
        df[f"dp_dt_{node}"] = (
            df.groupby("scenario_id")[pressure_col]
            .transform(lambda s: s.diff().fillna(0.0) / TIMESTEP_INTERVAL)
        )

    df["velocity_drop_ab"] = df["velocity_a"] - df["velocity_b"]
    df["velocity_drop_bc"] = df["velocity_b"] - df["velocity_c"]

    df["tke_drop_ab"] = df["tke_a"] - df["tke_b"]
    df["tke_drop_bc"] = df["tke_b"] - df["tke_c"]

    df["flow_velocity"] = df["velocity_b"]

    derived_cols = [
        "pressure_drop_ab", "pressure_drop_bc", "pressure_drop_ac",
        "pressure_gradient_ratio",
        "dp_dt_a", "dp_dt_b", "dp_dt_c",
        "velocity_drop_ab", "velocity_drop_bc",
        "tke_drop_ab", "tke_drop_bc",
        "flow_velocity",
    ]

    df[derived_cols] = df[derived_cols].fillna(0.0)

    print(f"  Derived features added : {len(derived_cols)}")
    print(f"  Final shape            : {df.shape}")
    print(f"  Columns present        : {list(df.columns)}")
    nan_counts = df[derived_cols].isna().sum()
    if nan_counts.any():
        print(f"  WARNING — NaNs remain  :\n{nan_counts[nan_counts > 0]}")
    else:
        print("  NaN check              : PASSED")

    return df

In [9]:
print("EXTRACTING SENSOR READINGS FROM ALL SCENARIOS")
print("=" * 60)

all_extracted = []

total = len(scenario_data)

for i, (df_scenario, scenario_id, label) in enumerate(scenario_data, 1):

    print(f"\n[{i}/{total}] Processing: {scenario_id}")

    extracted = extract_sensor_readings(
        df_scenario,
        scenario_id,
        label
    )

    print(f"  Extracted shape: {extracted.shape}")

    all_extracted.append(extracted)

print("\nCalculating derived features...")

df_all = pd.concat(all_extracted, ignore_index=True)

df_all = calculate_derived_features(df_all)
# Final checks
print("\nFINAL SHAPE:", df_all.shape)

print("\nLABEL DISTRIBUTION:")
print(df_all["label"].value_counts())
# SAVE OUTPUT
output_file = raw_data_path / "sensor_point_dataset_raw.csv"
df_all.to_csv(output_file, index=False)

print("\nDATA SAVED SUCCESSFULLY")
print("Saved to:", output_file)

EXTRACTING SENSOR READINGS FROM ALL SCENARIOS

[1/45] Processing: normal_baseline
[OK] normal_baseline → shape (700, 23)
  Extracted shape: (700, 23)

[2/45] Processing: normal_v105_c095
[OK] normal_v105_c095 → shape (700, 23)
  Extracted shape: (700, 23)

[3/45] Processing: normal_v100_c105
[OK] normal_v100_c105 → shape (700, 23)
  Extracted shape: (700, 23)

[4/45] Processing: normal_v095_c100
[OK] normal_v095_c100 → shape (700, 23)
  Extracted shape: (700, 23)

[5/45] Processing: normal_v096_c104
[OK] normal_v096_c104 → shape (700, 23)
  Extracted shape: (700, 23)

[6/45] Processing: normal_v098_c098
[OK] normal_v098_c098 → shape (700, 23)
  Extracted shape: (700, 23)

[7/45] Processing: normal_v105_c105
[OK] normal_v105_c105 → shape (700, 23)
  Extracted shape: (700, 23)

[8/45] Processing: normal_v095_c105
[OK] normal_v095_c105 → shape (700, 23)
  Extracted shape: (700, 23)

[9/45] Processing: normal_v104_c096
[OK] normal_v104_c096 → shape (700, 23)
  Extracted shape: (700, 23)

[

In [10]:
df_all.head(20)

,scenario_id,timestep,node_a_pressure,velocity_a,tke_a,tdr_a,wall_shear_a,tailings_vof_a,node_b_pressure,velocity_b,...,pressure_drop_ac,pressure_gradient_ratio,dp_dt_a,dp_dt_b,dp_dt_c,velocity_drop_ab,velocity_drop_bc,tke_drop_ab,tke_drop_bc,flow_velocity
0,blockage_25_loc16m,1,35793.499725,2.534141,0.034635,24.489446,11.197253,0.401012,19965.653843,2.474028,...,31403.077800,1.016219,0.000000,0.000000,0.000000,0.060113,-0.052493,-0.001492,-0.001854,2.474028
1,blockage_25_loc16m,2,36620.208522,2.484805,0.038275,25.658411,10.641124,0.405089,20589.416498,2.421521,...,32025.487923,1.002257,1653.417594,1247.525311,408.597347,0.063284,-0.089215,0.005690,-0.005282,2.421521
2,blockage_25_loc16m,3,36251.794105,2.473708,0.034279,25.294662,9.484428,0.398871,20946.810911,2.531422,...,32432.519693,0.893589,-736.828834,714.788826,-1550.892373,-0.057714,0.128389,-0.003798,0.003425,2.531422
3,blockage_25_loc16m,4,35694.151655,2.478258,0.032199,27.022206,9.489620,0.393614,19637.008191,2.406712,...,31795.602655,1.020249,-1115.284899,-2619.605441,158.549176,0.071546,-0.106356,-0.005860,0.004303,2.406712
4,blockage_25_loc16m,5,35834.383052,2.467021,0.037261,22.894842,10.660725,0.400522,19776.413014,2.442424,...,32793.646002,0.959505,280.462794,278.809648,-1715.623898,0.024597,-0.074601,0.002711,-0.003408,2.442424
5,blockage_25_loc16m,6,36787.297566,2.569537,0.035647,23.439768,10.250242,0.398309,19106.601335,2.521981,...,33068.072143,1.149039,1905.829027,-1339.623359,1356.976745,0.047556,-0.090364,0.000004,-0.002247,2.521981
6,blockage_25_loc16m,7,36144.736489,2.485088,0.034797,24.908137,10.644589,0.395470,20094.551930,2.449309,...,31567.301126,1.034354,-1285.122155,1975.901189,1716.419880,0.035779,-0.003084,-0.004033,0.002115,2.449309
7,blockage_25_loc16m,8,35566.008814,2.496601,0.034091,24.837599,10.843902,0.395164,19447.852780,2.483421,...,32307.184967,0.995622,-1157.455349,-1293.398300,-2637.223031,0.013180,-0.015963,-0.000884,-0.004864,2.483421
8,blockage_25_loc16m,9,36058.571722,2.555000,0.033721,24.339944,9.959091,0.393493,20766.372402,2.433175,...,32211.013981,0.903858,985.125815,2637.039245,1177.467786,0.121826,0.000501,0.004045,-0.005438,2.433175
9,blockage_25_loc16m,10,36087.612877,2.511439,0.037675,25.088742,10.973236,0.402177,19771.627334,2.479944,...,31701.560308,1.060473,58.082310,-1989.490136,1076.989657,0.031495,-0.060091,0.004455,0.001433,2.479944


In [12]:
print(f"\nLabel distribution :\n{df_all['label'].value_counts()}")


Label distribution :
label
2    10500
1    10500
0    10500
Name: count, dtype: int64
